# Amazon Sales Data Cleaning

Cleans `amazon_sales_data_uncleaned.csv` and writes `amazon_sales_data_cleaned.csv`
(listing-observation level), `amazon_products_deduped.csv` (one row per distinct
product), and `amazon_choice_daily.csv` (one row per product per day, built
specifically for modeling `is_amazons_choice` -- see the relevant sections near
the end for why three different granularities exist).

Decisions:
- `Rating` "X out of 5 stars" -> numeric `rating` (0-5 scale is implied, no need to keep unit text)
- `Number_of_reviews` "2,457" -> numeric
- `bought_in_last_month` "300+ bought in past month" -> numeric lower-bound estimate
- `Current/discounted_price`, `Listed_price` -> numeric, `$`/`No Discount` stripped
- `listed_price` NaNs filled with `current_price`: "No Discount" (0 true NaNs in the raw column) means there's no separate reference price, not that data is missing
- `current_price` NaNs (~28%) left as-is: 99% of these rows are out of stock, i.e. there's no active offer to price -- not recoverable, and `Price_on_variant` turned out to be a *different* variant's price, not a valid fallback. 142 rows are an exception (in-stock but genuinely no price) -- true unexplained missingness, still left as NaN.
- `final_price` added = `current_price` with the coupon applied (`coupon_pct`/`coupon_amount`), since coupons stack on top of `current_price` rather than being included in it. Guarded with an assertion that the two coupon fields are never both set (they aren't, but the formula would silently pick the wrong one if that ever changed).
- `Price_on_variant` dropped: 49% literal "nan" string, rest inconsistent (pack counts mixed with prices, and its $ values are for a different variant than the one priced in `current_price`)
- `image_url` dropped (not useful for analysis); `product_url` kept, but note 13% of rows are one-time `/sspa/click?...` ad-redirect tokens that never repeat even for the same product scraped twice -- not usable as an identifier for that subset. `Title` is the better (imperfect) identifier.
- `collected_at` parsed to datetime (spans 6 distinct days, not all the same day)
- `is_couponed` split into `has_coupon`, `coupon_pct`, `coupon_amount` (kept the discount magnitude instead of just a boolean)
- `is_best_seller` split into `is_best_seller`, `is_amazons_choice`, `has_deal_badge`, `deal_pct` (was a mixed promo-badge field, not a single flag)
- `sustainability_badges` split into `is_small_business` and `has_sustainability_badge` (badges like Small Business / Works with Alexa aren't sustainability signals)
- `category` added: keyword/regex classification of `Title` into 17 product categories (87% coverage, rest left as "Other"). Audio is checked before Phone & Tablet Accessories, since "for iPhone" compatibility copy on headphones/earbuds would otherwise misclassify them as phone accessories. Monitors & Displays excludes "display port" (with or without a space) so docking stations/desktops aren't misclassified as monitors.
- `brand` added: first word of `Title`, case-folded (reliable proxy -- Amazon titles are almost universally `Brand Model Description...`)
- exact duplicate rows dropped
- sanity-checked numeric extremes (`current_price`, `number_of_reviews`) and the missing rating/reviews pattern -- see the "Sanity checks" section near the end
- `is_amazons_choice` found to flicker within the same scrape day for 145/154 multi-scrape titles that ever flip -- treated as capture noise, not real badge changes (see the "(Title, day) daily aggregation" section)

An adversarial audit (separate agent) checked every claim above against the
full dataset rather than trusting spot-checks; findings that changed this
notebook are folded into the bullets above and the "Distinct-product view"
section. The one it flagged as most consequential: the listing-level file is
dominated by scrape-frequency of a handful of repeatedly-shown items, not
catalog diversity -- see that section before doing any row-level aggregate
(category/brand counts, avg price by category) on `amazon_sales_data_cleaned.csv`.


In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/amazon_sales_data_uncleaned.csv')
print(df.shape)
df.head()


(42675, 16)


,Title,Rating,Number_of_reviews,bought_in_last_month,Current/discounted_price,Price_on_variant,Listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,21-08-2025 11:14
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,21-08-2025 11:14
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,21-08-2025 11:14
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,21-08-2025 11:14
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,21-08-2025 11:14


## Drop exact duplicate rows

In [2]:
before = len(df)
df = df.drop_duplicates()
print(f'dropped {before - len(df)} duplicate rows')


dropped 962 duplicate rows


## Rating -> numeric

In [3]:
df['rating'] = df['Rating'].str.extract(r'([\d.]+)').astype(float)
df = df.drop(columns=['Rating'])
df['rating'].describe()


count    40689.000000
mean         4.391379
std          0.387622
min          1.000000
25%          4.200000
50%          4.500000
75%          4.600000
max          5.000000
Name: rating, dtype: float64

## Number_of_reviews -> numeric

In [4]:
df['number_of_reviews'] = (
    df['Number_of_reviews'].str.replace(',', '', regex=False).astype(float)
)
df = df.drop(columns=['Number_of_reviews'])
df['number_of_reviews'].describe()


count     40689.000000
mean       3046.495883
std       13100.081473
min           1.000000
25%          77.000000
50%         340.000000
75%        1845.000000
max      865598.000000
Name: number_of_reviews, dtype: float64

## bought_in_last_month -> numeric lower-bound estimate

e.g. `"300+ bought in past month"` -> `300`, `"1K+ bought in past month"` -> `1000`. `"No featured offers available"` and missing -> NaN for now (in-stock ones become 0 below, once `in_stock` exists).


In [5]:
def parse_bought(val):
    if not isinstance(val, str):
        return np.nan
    m = re.match(r'([\d.]+)(K)?\+', val)
    if not m:
        return np.nan
    num = float(m.group(1))
    if m.group(2) == 'K':
        num *= 1000
    return num

df['bought_in_last_month'] = df['bought_in_last_month'].apply(parse_bought)
df['bought_in_last_month'].describe()


count     31202.000000
mean        944.867316
std        4941.157670
min          50.000000
25%         100.000000
50%         200.000000
75%         400.000000
max      100000.000000
Name: bought_in_last_month, dtype: float64

## Prices -> numeric

In [6]:
def to_price(series):
    return pd.to_numeric(
        series.astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce',
    )

df['current_price'] = to_price(df['Current/discounted_price'])
df['listed_price'] = to_price(df['Listed_price'].replace('No Discount', np.nan))

# listed_price NaNs are not missing data: the raw column had zero true NaNs, every
# row was either a $ value or the literal "No Discount". "No Discount" means there is
# no separate strike-through reference price -- the listed price simply equals the
# current price. So this is filling in a known value, not imputing an unknown one.
df['listed_price'] = df['listed_price'].fillna(df['current_price'])

# current_price NaNs (~11.6k rows) are MOSTLY out-of-stock rows with no active offer
# to price (99% have buy_box_availability != 'Add to cart', ~0% coupons/sponsored).
# Exception: 142 rows are marked in-stock ("Add to cart") yet still have a genuinely
# blank raw price (e.g. Sony WH-CH720N, Epson EcoTank ET-8550) -- true missingness
# with no "out of stock" explanation, not covered by the story above. We checked
# Price_on_variant as a fallback source but it turned out to be the price of a
# *different* product variant (median ~15% off from current_price, sometimes 90%+
# off), so using it would silently inject wrong prices. Left as NaN either way:
# these rows still carry useful signal for availability/demand analysis, just not
# for price.
df = df.drop(columns=['Current/discounted_price', 'Listed_price', 'Price_on_variant'])
df[['current_price', 'listed_price']].describe()


,current_price,listed_price
count,30125.000000,30125.000000
mean,217.752181,237.113788
std,393.741077,431.719585
min,2.490000,3.590000
25%,29.690000,32.990000
50%,79.990000,85.990000
75%,199.990000,206.960000
max,4699.000000,5399.000000


## Coupon -> boolean + parsed savings

In [7]:
coupon = df['is_couponed']
df['has_coupon'] = coupon != 'No Coupon'
df['coupon_pct'] = coupon.str.extract(r'Save (\d+)%').astype(float)
df['coupon_amount'] = coupon.str.extract(r'Save \$([\d.]+)').astype(float)
df = df.drop(columns=['is_couponed'])
df[['has_coupon', 'coupon_pct', 'coupon_amount']].describe()


,coupon_pct,coupon_amount
count,1061.000000,887.000000
mean,18.442978,28.449076
std,14.556520,28.135681
min,5.000000,0.330000
25%,10.000000,16.000000
50%,15.000000,16.000000
75%,15.000000,25.000000
max,50.000000,133.000000


## final_price: current_price with the coupon applied

`current_price` and the clip-on-checkout coupon are two independent, stacking
discounts -- verified on rows with both a coupon and both prices present:
`current_price` was never equal to `listed_price` when a coupon existed, and the
coupon's size almost never matched the current-vs-listed gap. So `current_price`
is "the price before the coupon," not the final price a shopper actually pays.


In [8]:
both_set = df['coupon_pct'].notna() & df['coupon_amount'].notna()
assert not both_set.any(), (
    f"{both_set.sum()} rows have both coupon_pct and coupon_amount set -- "
    "final_price's two sequential .loc assignments assume mutual exclusivity "
    "and would silently drop one of them if this ever fires."
)

df['final_price'] = df['current_price']
df.loc[df['coupon_pct'].notna(), 'final_price'] = df['current_price'] * (1 - df['coupon_pct'] / 100)
df.loc[df['coupon_amount'].notna(), 'final_price'] = df['current_price'] - df['coupon_amount']
df['final_price'].describe()


count    30125.000000
mean       216.421206
std        392.731382
min          2.490000
25%         29.690000
50%         75.200000
75%        199.990000
max       4699.000000
Name: final_price, dtype: float64

## Other flag columns -> boolean

In [9]:
df['is_sponsored'] = df['is_sponsored'] == 'Sponsored'
df['in_stock'] = df['buy_box_availability'] == 'Add to cart'

# is_best_seller was a mixed promo-badge field, not just best-seller vs. not:
# split into distinct signals instead of collapsing everything else to False.
badge = df['is_best_seller']
df['is_best_seller'] = badge == 'Best Seller'
df["is_amazons_choice"] = badge == "Amazon's"
df['deal_pct'] = badge.str.extract(r'Save (\d+)%').astype(float)
df['has_deal_badge'] = (
    badge.isin(['Limited time deal', 'Ends in']) | df['deal_pct'].notna()
)

# sustainability_badges mixes true eco badges with unrelated ones
# (Small Business, Works with Alexa/Alexa Built-in, Made in Italy) -- split those out.
sustain = df['sustainability_badges']
eco_badges = {
    'Carbon impact', 'Energy efficiency', 'Manufacturing practices', 'Forestry practices',
    'Recycled materials', '1 sustainability certification',
}
df['is_small_business'] = sustain == 'Small Business'
df['has_sustainability_badge'] = (
    sustain.isin(eco_badges)
    | sustain.astype(str).str.startswith(('Energy efficiency +', 'Safer chemicals', 'Recycled materials +'))
)

df = df.drop(columns=['buy_box_availability', 'sustainability_badges'])
df[['is_sponsored', 'is_best_seller', 'is_amazons_choice', 'has_deal_badge', 'deal_pct', 'in_stock', 'is_small_business', 'has_sustainability_badge']].describe(include='all')

,is_sponsored,is_best_seller,is_amazons_choice,has_deal_badge,deal_pct,in_stock,is_small_business,has_sustainability_badge
count,41713,41713,41713,41713,336.000000,41713,41713,41713
unique,2,2,2,2,NaN,2,2,2
top,False,False,False,False,NaN,True,False,False
freq,35417,41438,40980,40920,NaN,27221,40372,40259
mean,NaN,NaN,NaN,NaN,20.372024,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,9.319615,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,9.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,12.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,18.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,30.000000,NaN,NaN,NaN


## Missing sales on in-stock listings -> 0 (i.e. "fewer than 50")

Amazon only shows "bought in past month" once a product sells at least 50
units in a rolling 30 days. So an **in-stock** listing with no badge sold
fewer than 50. Every value in this column is the lower bound of its bin
(`50` = 50-99), so `0` = the 0-49 bin -- the interval-censored sales model
reads it as [0, 50). (User's call: the difference between citing 0 and 50
is small, and 0 is the correct lower bound.)

**Out-of-stock** listings without the badge stay missing: the badge may be
hidden simply because nobody is selling the item right now, so we don't
know their sales.

In [10]:
below_50 = df['bought_in_last_month'].isna() & df['in_stock']
df.loc[below_50, 'bought_in_last_month'] = 0
print(f"{below_50.sum()} in-stock listing rows set to 0 (<50); "
      f"{df['bought_in_last_month'].isna().sum()} out-of-stock rows left missing")


4520 in-stock listing rows set to 0 (<50); 5991 out-of-stock rows left missing


## Sales bin upper bounds (`bought_upper`) and the "500+" heaping

`bought_in_last_month` is the bottom of a bin; the sales model needs the top
too. Amazon's ladder: 50+, 100+ ... 900+ (steps of 100), 1K+ ... 9K+ (steps
of 1K), 10K+ ... (steps of 10K). So normally `v` means [v, next step).

**Exception: "500+".** It holds far more products than its neighbours
(~1,100 vs ~80 each for 600+-900+), and 600+-900+ appear almost only in a
few categories (Printers & Ink, Office, Gaming, Batteries, Bags); in most
categories (Computers, TV, Storage, Keyboards, Car ...) there are ~0
products at 600-900 but plenty at 500. Distinct products at 500 vs 600-900:
Printers & Ink 51 vs 125, Office 18 vs 49, Gaming 8 vs 16, Batteries 37 vs
32; Computers 135 vs 2, Keyboards 67 vs 1, Storage 55 vs 0, TV 29 vs 0. So most categories display "500+"
for anything from 500 to 999 (matching Amazon's "category testing rules"
for this badge), and a few show the finer steps. The 100 / 1K / 10K bins
look big only because they're wider -- per unit of sales they're smooth.

Decision: **"500+" -> [500, 1000)** everywhere. In the fine-ladder
categories it really means [500, 600), so this is wider than necessary
there -- less informative, but never wrong. `0` (the <50 bin, in-stock only)
-> [0, 50); 100K+ -> no upper bound.

In [11]:
def upper(v):
    if np.isnan(v):
        return np.nan
    if v == 0:
        return 50.0
    if v == 50:
        return 100.0
    if v == 500:
        return 1000.0  # "500+" = 500-999 in most categories, see above
    if v >= 100_000:
        return np.inf
    step = 100 if v < 1000 else 1000 if v < 10_000 else 10_000
    return v + step

df['bought_upper'] = df['bought_in_last_month'].map(upper)
lo, hi = df['bought_in_last_month'], df['bought_upper']
assert ((hi > lo) | lo.isna()).all()
pd.crosstab(lo, hi).sum(axis=1).to_frame('rows').assign(upper=lambda t: t.index.map(upper))


,rows,upper
bought_in_last_month,,
0.0,4520,50.0
50.0,5967,100.0
100.0,8801,200.0
200.0,5602,300.0
300.0,2841,400.0
400.0,1436,500.0
500.0,2351,1000.0
600.0,186,700.0
700.0,102,800.0


## Note: Amazon Labor Day sale overlap

The scrape window (`2025-08-21` to `2025-08-30`) mostly overlaps Amazon's 2025 early Labor Day sale, which ran **2025-08-22 through 2025-09-01** ([USA Today, Aug 22 2025](https://www.usatoday.com/story/shopping/2025/08/22/amazon-early-labor-day-sale-2025/85759980007/)).

Only `2025-08-21` (~19% of rows) falls before the sale started; the other 5 collection days are all inside it. That makes a clean promo-vs-non-promo split uninformative here (it would really just be "is this the one pre-sale day"), so we deliberately did **not** engineer an `is_promo_day` flag. `collected_at` is kept as-is so this can be sliced manually if a result looks off.

## Drop image_url; parse collected_at to datetime

## category: keyword-based classification from Title

No product-category column exists in the raw data, and titles alone aren't
very informative on their own. This is a keyword/regex classifier over
`Title`, checked in priority order (most specific category first) -- picked
over TF-IDF/clustering (clusters still need manual labeling afterward, so it
doesn't remove the manual step) or an LLM call per title (real cost/complexity
for a step regex handles fine here, given how distinct this vocabulary is).

Keywords were built empirically from actual word frequency in the titles, not
guessed. Iterated a few rounds chasing the `Other` bucket down (18.1% -> 13.1%)
until further gains required either broad/risky keywords (e.g. bare `case`,
`radio`) or dozens of one-off keywords worth a handful of rows each --
diminishing returns, so left `Other` as a legitimate catch-all rather than
overfitting the keyword list to this one dataset.

**Revision: compatibility text and keyword collisions.** A hand-checked random
sample of 80 products (`data/category_gold_sample.csv`) put the first version
at 84% correct. The errors fell into three groups, each fixed here:

- **Compatibility text:** keywords matched anywhere in the title, so
  `"USB C Charger ... for MacBook"` landed in Computers. `categorize` now
  classifies the part of the title *before* the first `for` / `compatible
  with` / `works with` first, and only falls back to the full title if that
  part matches nothing. The fallback matters: for accessories the device
  after "for" *is* the category signal (`"OtterBox Case for iPhone 13"` has
  no keyword before "for"), so cutting it off unconditionally made things
  worse (79%).
- **Feature words matching an earlier category:** a mouse's "USB receiver"
  and a webcam's "stereo audio" hit Audio; a CB radio's or smartwatch's
  "LCD/AMOLED display" hit Monitors. Keyboards & Mice, webcams, and power
  banks are now checked before Audio, and `receiver` / `display` / `monitor`
  skip the common non-product uses (`USB receiver`, `LCD display`,
  `heart rate monitor`, `baby monitor`).
- **Vocabulary gaps:** CPUs by model (`Ryzen`, `i5-13400F`), PC power supplies
  (`ATX`, `80 Plus` -- bare "power supply" also matched UPS units and power
  strips), PC fans/coolers, `HDD`/`NAS`, access points (`UniFi`, `Omada`),
  `sound bar` as two words, photo `film`, `binding machine`, smart bands.

Result: 92.5% on the same gold sample, 711 products relabelled, `Other`
890 -> 818. Every rule was checked against a random sample of the products
it changed, not only the gold set, since 80 items is a small test.

**Revision 2: emptying `Other`.** `Other` (818 distinct products, 10%) is a
poor group for the analyses planned downstream: within-category price tiers
and category effects in a hierarchical model both assume a category is a
coherent product type, and `Other` mixed $5 pencils with $2,000 fish
finders. Clustering its titles by embedding showed three kinds of item:

1. **Electronics that belong in an existing category** but lacked a keyword
   (car amps/subwoofers/radios, lenses written as `28-70mm F2.8`, photo paper
   and scanners, SODIMM RAM, controllers, docks, Blu-ray drives/players,
   flashes/softboxes, office supplies like folders/staplers). Keywords added.
2. **Coherent groups with no category:** new `Marine & GPS` (fish finders,
   transducers, marine radios, satellite communicators), `Optics`
   (binoculars, telescopes, microscopes), `Car Electronics` (radar
   detectors, dash cams, double-DIN install kits). Home security sensors and
   doorbells went into `Smart Home & Wearables`, recordable media into
   `Storage & Memory Cards`. Small categories are fine here: partial pooling
   shrinks them rather than overfitting them, so there's no need to lump
   them together for sample size.
3. **Not consumer electronics** (bags, toys, personal care, kitchen, pet,
   furniture, hand tools): matched by `NON_ELECTRONICS`, labelled
   `Non-Electronics`, and dropped in the section below. This check only runs
   on titles no category claimed, so a device whose title merely mentions a
   backpack or a dog (AirTags, pet cameras) is never caught by it.

Each new rule was checked against a random sample of the products it moved,
including moves between existing categories (which caught e.g. `tempered
glass` pulling screen protectors into Computers, `amplifier` pulling a WiFi
repeater into Audio, `pencil` pulling iPad cases into Office). Gold sample:
93.75% (two gold labels were updated because the taxonomy changed: recordable
CDs -> Storage, a wake-up light -> Smart Home). `Other` 818 -> ~170 (2%),
now mostly one-offs (hard cases, electrical hardware, niche AV gear).

**Revision 3: `Bags & Cases`.** Categories should say what a product *is*,
not which device it's used with: a $60 laptop backpack filed under
Computers (median ~$140) reads as a "budget computer" in within-category
price tiers. Laptop bags/backpacks/sleeves, camera bags, Pelican/Nanuk hard
cases, and tech organizers (~110 products, mostly out of Computers) now get
their own category, checked first. Only bags whose title names a device
qualify; generic backpacks stay non-electronics and are dropped. Phone and
tablet cases stay in `Phone & Tablet Accessories`, which is explicitly an
accessories category. The broader device-vs-accessory split (laptop stands
and docks in Computers, tripods in Cameras) is left as a possible
`is_accessory` flag -- see `HANDOFF.md`.

**Revision 4: devices vs accessories, and the user's labeling rules.** Hand
labels of 250 random products (see "Label corrections" below) showed the
biggest remaining error: renewed iPhones/iPads, Galaxy tablets, Kindles and
cordless home phones filed as *accessories* (the `iphone|galaxy|tablet`
keywords claimed them). New and changed rules, each checked by reading the
products it moved:
- `Phones & Tablets` (new, ~190 products): the device itself -- the title
  starts with the device name (after brand/"Renewed"/year words) and has no
  accessory noun (case, charger, stand, ...), or names a cordless/DECT/VoIP
  phone. Accessories stay in `Phone & Tablet Accessories`.
- Smartwatches and fitness bands (Galaxy Watch/Fit, "smartwatch") are
  checked early so "Galaxy" or "Heart Monitor" no longer claim them.
- All SSDs (internal and external) -> `Storage & Memory Cards`, with hard
  drives; laptops/PCs that list an SSD spec still go to Computers.
- Anything sold for use in a car (car chargers, car amps/speakers/stereos,
  FM transmitters) and radio scanners -> `Car Electronics`: they're bought
  in the context of the car (user's call).
- Video production / conferencing gear (ATEM switchers, Blackmagic
  converters, Rally Bars) -> `Cameras & Photography`.

**Revision 5 (2026-09-25, from the price-model error review).** ~40 UPS
battery backups were scattered over Batteries, Cables & Charging and TV
(their titles say "battery", "USB charging ports", "TV & computer"). They now
go to Power Strips & Surge Protectors (user's call, matching the guidelines);
UPS *replacement battery cartridges* go to Batteries. Also: smart locks ->
Smart Home, conference-room video systems (Logitech MeetUp, Rally Plus) ->
Cameras. Reference labels made before this are re-mapped by `to_current`
(this changes one user label: an APC UPS the user had put in Batteries).

**Revision 6 (2026-09-25).** The last misfiles from that review: corded and
senior home phones ("Corded/Cordless Senior Phone", "Corded Telephone",
"Corded Answering System") were accessories -> `Phones & Tablets`, with the
other home phones; the Poly Studio X52 video bar was Audio -> Cameras, with
the other video bars; a CompactFlash card (Cameras, via "flash") and a MiniDV
tape (Audio, via "cassette") -> `Storage & Memory Cards`, with the other
memory cards and Hi8 tapes; bathroom exhaust fans (one in Car Electronics via
"installation kit") are not electronics and are dropped.
Re-running also surfaced a few products whose label had depended on the
category model's fold split: an SDHC card (Cameras via "trail camera"), a
weather-alert radio (Monitors via "display"), an Elgato HD60 capture card
(Gaming via "PS5") and a car charger (Computers via "MacBook"; car items in
Computers now go to Car Electronics like the other car items). Each got a
rule.

In [12]:
# Bags and hard cases made for electronics: a laptop backpack is an accessory,
# not a laptop, and would otherwise be filed by its device word (Computers,
# Cameras). Requires a device word somewhere in the title, and the bag noun
# before any "with"/"bundle" (so "Camera Bundle with Bag" stays a camera).
# Generic bags with no device word fall through to Other -> Non-Electronics.
BAGS = (
    r"^(?!.*(airtag|otterbox|grab bag))"
    r"(?=.*\b(laptops?|notebooks?|computers?|macbook|chromebook|tablets?|ipad|cameras?|dslr|slr|mirrorless|drones?|electronics|tech)\b)"
    r"(?:(?! with | w/ |bundle| \+ ).)*?(backpack|\bbag\b|\bsleeve\b|laptop case|briefcase|\btote\b|sling bag|everyday sling|messenger bag)"
    r"|pelican (air )?\d{4}|nanuk|electronics organizer"
)
# Phones and tablets themselves (incl. renewed), not their accessories: the
# title starts with the device name (after brand / "Renewed" / year words) and
# has no accessory noun. Checked early: tablet titles mention "RAM"/"Display".
PHONES = (
    r"^(?!.*\b(cases?|covers?|protectors?|chargers?|cables?|stands?|holders?|mounts?|stylus|keyboard|folio|skins?|straps?|bands?"
    r"|adapters?|docks?|grips?|lens|tripod|gimbal|earbuds|headphones|headsets?|expansion|wallet|sleeve|bag|replacement|parts?|tool|s[- ]?pen for)\b)"
    r"(?:(?:renewed|new|\(renewed\)|\d{4}|apple|samsung|google|motorola|oneplus|amazon|lenovo|tcl|nokia)\W+)*"
    r"(?:iphone|ipad|galaxy (?:s\d|a\d|z ?(?:fold|flip)|tab|note)\w*|pixel \d|kindle|moto [gez]|oneplus (?:pad|\d+|nord)|fire (?:hd )?\d+\b|tab [a-z]?\d)"
    r"|\b(?:unlocked smartphone|android tablet|cordless phone|dect 6\.0|corded phone|voip phone|ip phone|desk phone|landline)\b"
    r"|(?:corded|cordless)[\w/ -]{0,20}?\b(?:tele)?phone\b|corded answering system"  # "Corded/Cordless Senior Phone", "Corded Telephone"
)
# Smartwatches / fitness bands whose titles would otherwise hit "galaxy" or
# "heart monitor" first. Bands and chargers for them are excluded.
WATCHES = r"^(?!.*\b(bands?|straps?|chargers?|cases?|protectors?|charging)\b).*(galaxy (?:watch|fit|ring)|\bsmartwatch\b|smart watch)"
# SSDs are storage whether internal or external (like hard drives); laptops
# and PCs that list an SSD spec are excluded (they still reach Computers via \bssd\b).
SSD = r"^(?!.*\b(laptop|notebook|ultrabook|desktop|mini pc|gaming pc|computer|chromebook|chromebox|macbook|mac mini|mac studio|surface|thinkpad|workstation|tablet|tab|ipad|console|handheld|ram|cpu|processor|intel|ryzen|celeron|windows|motherboard|mini-itx|microatx|heatsink|bracket|hub)\b).*(\bssd\b|solid state drive|\bnvme\b)"
# Video production / conferencing gear goes with cameras (user's call).
VIDEO = r"webcam|\batem\b|blackmagic|rally (?:bar|plus)|video ?conferenc|conference (camera|room)|meetingbar|\bmeetup\b|capture card|cam link|game capture|\belgato hd60\b"
# UPS battery backups are surge protectors with a battery (user's call, as in the
# guidelines); their replacement battery cartridges are batteries. Checked early:
# UPS titles also say "battery", "USB charging ports", "TV & computer".
UPS_BATTERY = r"\bups\b.*\b(replacement battery|battery replacement|battery cartridge)\b|\b(replacement battery|battery replacement)\b.*\bups\b"
UPS = r"^(?!.*\b(label printer|rails?|calculator)\b).*(\bups\b|uninterruptible)"
SMART_LOCK = r"smart lock|smart entry|smart module|(deadbolt|door lock).*(wi-?fi|keypad|fingerprint)|(wi-?fi|keypad|fingerprint).*(deadbolt|door lock)"
# Conference-room video systems also list microphones/speakers/PC compatibility.
VIDEO_SYSTEM = r"\bmeetup\b|rally (?:bar|plus)|video conferencing system|conference room camera|video bar"
# Blank camera media: "Compact Flash Card" otherwise hits the camera "flash"
# keyword, "Video Cassette" the audio "cassette" keyword, "SDHC Trail Camera
# Flash Memory" the camera "trail cam" keyword.
CAMERA_MEDIA = r"video cassette|\bminidv\b|compact ?flash|\bcf card|^(?!.*bundle).*\b(sdhc|sdxc)\b"
# Household ventilation fans; one title also says "Installation Kit" (a car keyword).
HOME_FAN = r"exhaust fan|ventilation fan"
# "Trilingual Display" otherwise sends a weather radio to Monitors.
WEATHER_RADIO = r"weather (alert )?radio"
CATEGORY_PATTERNS = [
    ("Bags & Cases", BAGS),
    ("Phones & Tablets", PHONES),
    ("Smart Home & Wearables", WATCHES),
    ("Storage & Memory Cards", SSD),
    ("Batteries", UPS_BATTERY),
    ("Power Strips & Surge Protectors", UPS),
    ("Smart Home & Wearables", SMART_LOCK),
    ("Cameras & Photography", VIDEO_SYSTEM),
    ("Storage & Memory Cards", CAMERA_MEDIA),
    ("Non-Electronics", HOME_FAN),
    ("Audio", WEATHER_RADIO),
    ("Protection Plan", r"asurion|protection plan|complete protect|\d+ year (home|major appliance)"),
    ("Printers & Ink", r"\bprinter\b|\bink\b|toner|cartridge|laserjet|\bmfp\b|photo paper|presentation paper|project paper|business paper|matte paper|bond paper|printhead|drum unit|\bscanjet\b|flatbed|document scanner|photo scanner|(?<!barcode )(?<!digital )(?<!radio )\bscanner\b|fax machine"),
    ("Office & School Supplies", r"\blabels?\b|\bavery\b|\bdymo\b|calculator|texas instruments|barcode scanner|mailing label|printable label|laminat|shredder|post-it|sticky note|binding (machine|comb)"
                                 r"|file folders?|hanging (file )?folders?|folder|\bstapler\b|\bstaples\b|\bpencils\b|(mechanical|drafting|#2) pencil|envelopes?|binder clips|dry[- ]erase|whiteboard|copy paper|index cards|card ?stock|sheet protector|tape dispenser|packing tape|\bmarkers\b|clipboard|desk organizer|lined notebook|note pads|\bpapers \|"),
    ("Computers & Laptops", r"laptop|desktop|macbook|chromebook|\bssd\b|\bram\b|motherboard|\bcpu\b|processor|\bpcie\b|\bnvme\b|docking station|(?<!charging )(?<!mouse )\bdock\b|cooling fan|\bpc fans?\b|\bgpu\b|graphics (card|board)|\brtx\b|\bgtx\b"
                            r"|ryzen|\bcore i[3579]\b|\bi[3579]-\d{4,5}|\batx\b|80 ?\+|80 plus|modular power supply|\d{3,4}w\b.*power supply|\bpsu\b|argb fan|pwm (reverse )?fans?|fan hub|\d{3}mm fan|cpu cooler|liquid cooler|sodimm|\bddr[345]\b|\bquadro\b"
                            r"|computer case|mid-tower|pc case|\bjetson\b|arduino|raspberry pi|microcontroller|breadboard|developer kit"),
    ("Smart Home & Wearables", r"digital (picture|photo) frame|(picture|photo) frame.*wi-?fi|wi-?fi.*(picture|photo) frame"),  # user's call
    ("Monitors & Displays", r"(?<!baby )(?<!rate )(?<!reference )(?<!tv )\bmonitor\b|(?<!lcd )(?<!led )(?<!oled )(?<!backlit )(?<!color )(?<!touch )\bdisplay\b(?!\s?port)"),
    # Checked before Audio: mouse/keyboard titles mention a "USB receiver",
    # webcams advertise "stereo audio", and power banks mention iPhone/battery,
    # all of which match keywords in categories checked earlier.
    # tablet keyboard cases are tablet accessories (Surface covers stay with Computers, user's call)
    ("Phone & Tablet Accessories", r"^(?=.*\bkeyboard\b)(?=.*\b(case|folio|cover)\b)(?=.*\b(ipad|tablet|galaxy tab|fire hd|fire max)\b)"),
    ("Keyboards & Mice", r"\bkeyboard\b(?!\s*(case|folio|cover))|\bmouse\b|mouse ?pad|trackpad|wrist rest|palm rest|stream deck|unifying receiver"),
    ("Cameras & Photography", VIDEO),
    ("Cables & Charging", r"power bank|portable charger"),
    ("Smart Home & Wearables", r"wake-up light"),
    ("TV & Home Entertainment", r"projector(?! mount)"),
    ("Marine & GPS", r"fish ?finder|transducer|chartplotter|depth sounder|marine radio|humminbird|lowrance|satellite communicator|\binreach\b"),
    ("Car Electronics", r"radar detector|radar/laser|laser/radar|laser detection|double din|installation kit|wiring harness|dash ?cam|smartcord|car antenna|antenna compatible with"),
    ("Optics", r"binocular|telescope|monocular|microscope|eyepiece|spotting scope|observatory|rangefinder"),
    # Audio checked before Phone & Tablet Accessories: audio product titles very
    # commonly mention "for iPhone"/"Phone Calls" compatibility, which would
    # otherwise misclassify headphones/earbuds as phone accessories (verified:
    # 189/1,669 "headphone" titles, incl. AirPods/Sony WH-1000XM4, were wrongly
    # landing in Phone & Tablet Accessories before this reorder).
    ("Audio", r"speaker|headphone|headset|earbud|airpods|microphone|\bmic\b|sound ?bar|\bstereo\b|two-way radio|walkie[- ]talkie|turntable|av receiver|(?<!usb )(?<!nano )(?<!unifying )(?<!bolt )(?<!ghz )\breceiver\b|cb radio"
              r"|(?<!repeater )(?<!antenna )(?<!signal )\bamplifier|subwoofer|bass module|earphones|homepod|\bradio\b|voice recorder|mp3 player|\bdac\b|stylus \d|replacement stylus|record (brush|player)|vinyl|surround sound|cassette|packtalk|sustain pedal|\bsonos\b|\bcomponents\b"),
    ("Phone & Tablet Accessories", r"keyboard (case|folio|cover)|iphone|\bipad\b|galaxy|\bphone case\b|screen protector|phone mount|phone holder|\bphone\b|\btablet\b|\bkindle\b|e-?reader|\bpixel \d|airtag|popsocket|poptop|moto tag|apple pencil|stylus|active pen|tilt pen|\bpen\b"),
    ("Cameras & Photography", r"\bcamera\b|\bcam\b|\blens\b|\bcanon\b|\bnikon\b|gopro|tripod|monopod|\bdslr\b|fujifilm|\binstax\b|polaroid|\bfilm\b|\b135-\d{2}\b|\d+-\d+mm f/?\d|\d+mm f/?\d|\bf\d\.\d\b|\bdg dn\b|teleconverter"
                              r"|speedlite|\bflash\b(?!\s*(drive|memory))|strobe|softbox|light dome|diffus|bowens|key light|video light|ring light|tube light|selfie stick|backdrop|photo studio|bounce card|trail cam|capture card|video recorder|camcorder|\bdrone\b|gimbal|stabilizer"),
    ("Cables & Charging", r"hdmi (fiber|optic)"),
    ("Networking", r"\bwifi\b|wi-fi|router|range extender|ethernet|\bmodem\b|\bnetgear\b|\bpoe\b|server rack|access point|\bunifi\b|\bomada\b|network switch|patch panel|keystone|\brj45\b|nanobeam|cloud gateway"),
    ("Gaming", r"\bxbox\b|playstation|\bps5\b|\bps4\b|nintendo|gaming controller|\bgamepad\b|(?<!with )(?<!rgb )(?<!remote )\bcontroller\b|joy-con|amiibo|meta quest|\bhosas\b|flight (pack|stick)|racing (cockpit|wheel)|simulator cockpit|arcade|joystick|game & watch|8bitdo"),
    ("TV & Home Entertainment", r"\btv\b|remote control|streaming device|\bfire tv\b|\broku\b|projector|projection screen|blu[- ]?ray (dvd |disc )?player|dvd player|\bhdtv antenna|indoor antenna|universal remote"),
    ("Storage & Memory Cards", r"memory card|microsd|flash drive|\bhard drive\b|external drive|\bsdhc\b|\bsdxc\b|memory flash|\bhdd\b|\bnas\b|diskstation|cfexpress|card reader|dvd[+-]r|cd-r|recordable|rewritable|blu-ray (drive|burner)|read/write|\bburner\b|optical drive|dvd/cd|dvd rw|bdxl|\bdiscs?\b"),
    ("Smart Home & Wearables", r"\balexa\b|\becho\b|smart plug|\bnest\b|\bring\b doorbell|doorbell|smart ?watch|smart band|fitness (tracker|band)|activity[- ]tracker|\bgps\b|cycling computer|\bhrm\b|watch band|band for apple watch|quickfit|\bfitbit\b"
                               r"|motion sensor|entry sensor|door sensor|window and door|security system|simplisafe|wake-up light|electronic safe|home safe"),
    ("Batteries", r"\bbatteries\b|\bbattery\b|\blithium\b.*\bcell\b|\benergizer\b|\bduracell\b|\bcell pack\b|3v lithium"),
    ("Power Strips & Surge Protectors", r"surge protector|power strip|extension cord|\boutlet\b|power conditioner|timer switch|wall ?plate|circuit breaker|surgebreaker|gfci|light switch|power distribution|\bpdu\b|surge protective"),
    ("Cables & Charging", r"\bcable\b|\bcharger\b|charging|\bhdmi\b|\busb-c\b|power adapter|power bank|\badapter\b|\bups\b|\bhub\b|cord (bundlers|organizer)|connectors?\b|crimp terminal|tie wire"),
]
# Not consumer electronics: bags, toys, personal care, household, kitchen,
# pet, furniture, hand tools, apparel. Only applied to titles that no category
# above claimed (i.e. would otherwise be "Other"), so a device title that
# merely mentions a backpack/dog/wallet is never caught by it.
NON_ELECTRONICS = re.compile(r"backpack|\btote\b|daypack|shoulder bag|travel pack|\bpuzzle|\btoys?\b|play food|pretend play|melissa & doug|crayola|oil pastel|\bgame \(\d+ pcs"
                        r"|deodorant|antiperspirant|mouthwash|hand ?wash|hand sanitizer|ointment|facial tissue|paper towels|car wash|car cleaning|cleaning wipes|screen wipes|screen cleaner|wipes for car|cleaning cloth|chamois|respirator"
                        r"|\bdogs?\b|\bpuppy\b|\bpet\b|bakeware|loaf pan|paring knife|pepper mill|toaster|kettle|hand mixer|multi-cooker|\bopener\b|coffee maker and espresso machine cleaner|cooktop"
                        r"|desk chair|task chair|mid-back|\bfoot rest\b|storage shed|wire rack|patio heater|humidifier|lantern"
                        r"|screwdriver|wire cutter|circular saw|staple gun|combination lock|deadlatch|heat shrink|velcro"
                        r"|guitar strings|guitar stand|ukulele|wristband|sweatbands|dri fit|shaver|\brazor\b|beard trimmer|haircut|hearing protector|soil color book|world's most amazing|play slide|remote starter|mobile wallet|golf", re.I)

# Anything sold for use in a car goes in Car Electronics (user's call: car
# chargers, car amps, FM transmitters, radio scanners are bought for the car).
# Applied after the keyword match, to titles that landed in a generic category.
CAR = re.compile(r"\bcar (audio|stereo|speakers?|amplifier|amp|charger|radio)\b|\bcar\b.*\bcharger\b|cigarette lighter|fm transmitter"
                 r"|\bscanner\b.*\b(police|trunk ?tracker|channels|bearcat|beartracker)\b|\b(police|bearcat|beartracker|uniden)\b.*\bscanner\b|radio scanner", re.I)
CAR_FROM = {"Audio", "Cables & Charging", "Printers & Ink", "Computers & Laptops", "Other"}  # Computers: "... Car Charger for ... MacBook"

# Plain stationery is not electronics (user's call, 2026-09-25): dropped with the
# non-electronics. Kept in Office & School Supplies: the devices (calculators,
# label makers, shredders, laminators, presenters), their supplies (label tapes,
# laminating pouches, binding combs) and printable paper/cards/labels, which are
# printing supplies like photo paper.
STATIONERY = re.compile(r"\bbinders?\b|binder clips|folders?\b|\bpencils?\b|\bpens?\b|\bmarkers?\b|sticky notes|post-it"
                        r"|note ?pads?|\bnotebook\b|writing pad|index cards|\bstaplers?\b|\bstaples\b|staple gun|whiteboard"
                        r"|white board|dry[- ]erase|clipboard|envelopes?|dividers|sheet protectors?|organizer|easel pad"
                        r"|coloring|name badges?|name tags?|tab inserts|spine inserts|expanding file", re.I)
OFFICE_DEVICE = re.compile(r"calculator|label ?maker|labelmanager|labelwriter|printer\b|laminat|shredder|presenter|electric"
                           r"|usb|bluetooth|battery|digital|wireless|barcode|scanner|air duster|compressed air|\bcd\b|\bdvd\b"
                           r"|binding (machine|comb)", re.I)
PRINT_SUPPLY = re.compile(r"printable|\blabels?\b|tape|pouch|p-touch|\btze\b|\bpaper\b|card ?stock", re.I)
ALWAYS_STATIONERY = re.compile(r"ring binders?|economy view|easel pad", re.I)

def is_stationery(title):
    if OFFICE_DEVICE.search(title):
        return False
    return bool(ALWAYS_STATIONERY.search(title)) or (bool(STATIONERY.search(title)) and not PRINT_SUPPLY.search(title))

COMPAT = re.compile(r'\s(?:for|compatible with|works with)\s', re.I)
DROP_CATEGORIES = {"Protection Plan", "Non-Electronics"}

def _match(text):
    t = text.lower()
    for cat, pattern in CATEGORY_PATTERNS:
        if re.search(pattern, t):
            return cat
    return "Other"

def categorize(title):
    head = COMPAT.split(title, maxsplit=1)[0]
    cat = _match(head) if len(head) >= 15 else "Other"
    cat = cat if cat != "Other" else _match(title)
    if cat in CAR_FROM and CAR.search(title):
        return "Car Electronics"
    if cat == "Office & School Supplies" and is_stationery(title):
        return "Non-Electronics"
    if cat == "Other" and NON_ELECTRONICS.search(title):
        return "Non-Electronics"
    return cat


df['category'] = df['Title'].apply(categorize)
df['category'].value_counts()

category
Audio                              6117
Computers & Laptops                5576
Cameras & Photography              4335
Protection Plan                    3068
Printers & Ink                     2962
Cables & Charging                  1956
Batteries                          1727
Phone & Tablet Accessories         1653
Monitors & Displays                1604
Networking                         1461
Other                              1356
Smart Home & Wearables             1349
Keyboards & Mice                   1220
Storage & Memory Cards             1163
Gaming                             1083
Non-Electronics                     821
Office & School Supplies            812
Optics                              688
Car Electronics                     679
Phones & Tablets                    479
TV & Home Entertainment             442
Power Strips & Surge Protectors     428
Marine & GPS                        402
Bags & Cases                        332
Name: count, dtype: int64

## brand: first word of Title, skipping non-brand leading tokens

Amazon listing titles are almost universally `Brand Model Description...`,
so the first word is a reliable brand proxy in most cases. But some titles
lead with a spec/quantity instead (`"2-Pack LR44..."`, `"67XL Ink
Cartridges..."`, `"(2) Kicker..."`, `"2018 Apple iPad..."`) -- these weren't
caught by the earlier "rare first-word" check, since many different titles
share the same bad leading token (e.g. many pack-count listings independently
start with `"2-Pack"`), so it doesn't look rare in a frequency count even
though it's structurally wrong every time.

Tried and rejected: an allow-list bootstrapped from words that frequently
lead *other* titles ("trusted brand vocabulary"), scanning each title's first
few words for a match. This backfired -- it picks up bare numbers/generic
words that happen to lead >=3 titles for unrelated reasons (`"2"`, `"Case"`,
`"New"` all got "trusted"), and worse, it opportunistically grabs an
unrelated brand mentioned deeper in a title as a compatibility/component
note (e.g. `"CyberPowerPC ... AMD Ryzen..."` -> wrongly became `"Amd"`;
`"Ray-Ban Meta Glasses"` -> wrongly became `"Meta"`). That's a bigger error
class than the one it was fixing, because it can silently override rows
that were already correct.

Used instead: a **narrow blacklist of specific shapes** (bare numbers,
number+unit like `12-Pack`/`256GB`/`67XL`, punctuation-only tokens) that only
*triggers* a fix when the very first word matches one of these shapes --
it can never touch a row where the first word was already fine, so it can
only fix things, not introduce new errors. When triggered, it scans forward
(up to 5 words) for the first word that isn't also one of these shapes.
Affects 42/41,713 rows (0.1%) -- small, but these were previously visibly
wrong (`"2-Pack"`, `"(2)"`, `"128Gb"` are not brands). A few residual misses
remain (e.g. `"HP 67XL Ink..."`-style titles resolve to `"Ink"` rather than
`"HP"`, since "HP" isn't the very next word either) -- accepted as
diminishing returns, same as the `category` classifier's `Other` bucket.

Known coarse-graining: "Amazon" as first word lumps Amazon Basics (private
label commodity items) together with Kindle/Fire (Amazon's own device
lines) -- a small number of rows, but worth knowing if `brand == "Amazon"`
ever looks like an outlier in an analysis.

This lets category-level price/rating variation be split further into a
brand effect vs. a within-brand effect (e.g. "is Audio pricier than
Batteries" vs. "is Sony pricier than a no-name brand within Audio").

**Normalization pass** (added after inspecting brand counts for near-duplicates):
the same brand was showing up under several spellings, splitting its products
across brand groups -- `Intel` (21) vs `Intel®` (18), `Garmin`/`Garmin®`,
`Brother`/`Brother(R)`, `Midland`/`Midland-`/`Midland®`, and a `Popsockets`
prefixed with invisible zero-width spaces. Trademark marks, zero-width
characters, and trailing hyphens are stripped before taking the first word.

Separately, first-word extraction truncates multi-word brands to a generic
word (`Texas` for Texas Instruments, `Western` for Western Digital, `The` for
The North Face, `Cooler` for Cooler Master). `MULTI_WORD_BRANDS` checks the
start of the title for a short list of these, found by looking for brand
tokens that are a prefix of another brand or a common English word, and
maps a few one-off spellings (`CanonCL-261`, `TI-84`, `Boyalink`) to their
parent brand. Not exhaustive -- it covers every case with more than one
product plus the obvious one-offs; long-tail misses stay as-is.

**Generic first words and `Unbranded`.** Some titles start with a descriptor
instead of a brand ("4Pc Size C Panasonic Batteries", "Genuine Canon Ink",
"64XL Ink Cartridges ... Remanufactured for HP"). When the first word is in
`GENERIC_FIRST`, the brand is the first *known* brand (one that starts 3+
titles) later in the title, looking only before "for / compatible with" --
otherwise "Remanufactured for HP" would credit HP with a third-party
cartridge. If none is found the brand is `Unbranded` (~35 products):
generic / white-label listings that don't name their maker in the title
(the seller's brand is often only on the product page). Treat `Unbranded`
as its own group in models -- it's a meaningful kind of listing, not
missing data -- but expect it to be heterogeneous.

**Sub-brands (2026-09-25).** Product lines and spelling variants of one maker
are merged into the maker: WD_BLACK / WD -> Western Digital, ROG / TUF ->
Asus, Legion -> Lenovo, Predator -> Acer, Joy-Con -> Nintendo, Oculus -> Meta,
Hewlett-Packard -> HP. Brands that shoppers see as brands in their own right
(Alienware, Beats, Xbox, PlayStation, Ring, Blink, eero, Soundcore, HyperX)
stay separate, following the Amazon-brand rule.

In [13]:
NON_BRAND_SHAPE = re.compile(
    r'^\(?-?\d+(-\d+)?\)?$|^\d+-?(gb|tb|mb|mm|xl|pk|pc|pcs|ct|cou|pack)$|^(x|count|pcs|pc|ct|new|the)$|^[^a-z0-9]+$',
    re.I,
)

# title-prefix pattern -> canonical brand, for brands the first word would truncate or misspell
MULTI_WORD_BRANDS = {
    r'amazon ?basics': 'Amazon Basics',  # Amazon's private label, distinct from Amazon devices (Echo, Kindle)
    r'texas instruments|ti[- ]?\d\d\w*': 'Texas Instruments',
    r'western digital': 'Western Digital',
    r'the north face': 'The North Face',
    r'cooler ?master': 'Cooler Master',
    r'lian[- ]li': 'Lian Li',
    r'nite ize': 'Nite Ize',
    r'mounting dream': 'Mounting Dream',
    r'insta ?360': 'Insta360',
    r'audio[- ]technica': 'Audio-Technica',
    r'vic firth': 'Vic Firth',
    r'sound ?blaster\w*': 'Sound Blaster',
    r'game & watch': 'Nintendo',
    r'ring fit': 'Nintendo',
    r'stealth ?cam': 'Stealth Cam',
    r'sig sauer': 'Sig Sauer',
    r'star wars': 'Star Wars',
    r'sirius-?xm': 'Siriusxm',
    r'on-?stage': 'On-Stage',
    r'pyle-pro': 'Pyle',
    r'boyalink': 'Boya',
    r'canoncl-': 'Canon',
    r'c2g': 'C2G',
}
MULTI_WORD_RE = [(re.compile(rf'^({pat})\b', re.I), brand) for pat, brand in MULTI_WORD_BRANDS.items()]
MARKS = re.compile(r'[​‌‍﻿®™©]|\((r|tm)\)', re.I)

# Descriptor words that start some titles in place of a brand ("Wireless
# Mouse", "Genuine Canon Ink", "64XL Ink Cartridges ... for HP"). When the
# first word is one of these, look for a known brand later in the title
# (before any "for / compatible with", so "for HP" never makes it HP);
# failing that the listing is Unbranded -- generic/white-label products that
# don't name a maker in the title.
GENERIC_FIRST = {
    'wireless', 'usb', 'usb-c', 'bluetooth', 'battery', 'batteries', 'laptop', 'gaming', 'power', 'case', 'ink',
    'desktop', 'waterproof', 'water', 'a', 'genuine', 'original', 'oem', 'bundle', 'replacement', 'rechargeable',
    'size', 'super', 'matte', 'aa', 'aaa', 'industrial', 'elite', 'lr44', 'cr', '3x', 'ups', 'compatible',
    'remanufactured', 'universal', 'portable', 'premium', 'mini', 'digital', 'smart', 'outdoor', 'all-new', 'kids', 'breadboard', 'breadboards',
}
# Tokens with digits are model codes (AX3000, B450M-HDV), except these brands
DIGIT_BRANDS = {'8bitdo', '3m', 'ds18', '1more'}
LENS_SHAPE = re.compile(r'^\d+(-\d+)?mm$', re.I)

def first_word(title):
    title = MARKS.sub('', str(title))
    # drop a leading bracketed tag, e.g. "[Older Version] SanDisk ..."
    title = re.sub(r'^\[[^\]]*\]\s*', '', title).strip()
    # "Made for Amazon" is a licensing program, not the maker ("Made for Amazon SanDisk ...")
    title = re.sub(r'^made for amazon,?\s*', '', title, flags=re.I)
    for pat, brand in MULTI_WORD_RE:
        if pat.match(title):
            return brand
    words = [w.strip(',').rstrip('-') for w in title.split()]
    for w in words[:5]:
        if w and not NON_BRAND_SHAPE.match(w):
            return w
    return words[0] if words else ''

def extract_brand(title, known):
    b = first_word(title)
    model_code = re.search(r'\d', b) and b.lower() not in DIGIT_BRANDS and b not in MULTI_WORD_BRANDS.values()
    if b.lower() not in GENERIC_FIRST and not LENS_SHAPE.match(b) and not model_code:
        return b
    head = COMPAT.split(MARKS.sub('', str(title)), maxsplit=1)[0]
    for w in re.split(r'[\s,]+', head)[1:]:
        w = w.strip(',:;()-').lower()
        if w in known:
            return w
    return 'Unbranded'

first = df['Title'].drop_duplicates().apply(first_word)
counts = first.str.lower().value_counts()
known = set(counts[counts >= 3].index) - GENERIC_FIRST

assert extract_brand('4Pc Size C Panasonic Batteries', known) == 'panasonic'
assert extract_brand('64XL Ink Cartridges Remanufactured for HP Ink 64', known) == 'Unbranded'
assert extract_brand('Genuine Canon PG-240XL', known) == 'canon'
assert extract_brand('2025 AX3000 Outdoor WiFi 6 Extender, WAVLINK Long-Range', known) == 'wavlink'
assert first_word('Intel® Core i9') == 'Intel'
assert first_word('​​PopSockets Grip') == 'PopSockets'
assert first_word('Midland- T51VP3') == 'Midland'
assert first_word('[Older Version] SanDisk 128GB') == 'SanDisk'
assert first_word('Texas Instruments TI-84') == 'Texas Instruments'
assert first_word('TI-84 Plus CE') == 'Texas Instruments'
assert first_word('2-Pack Duracell AA') == 'Duracell'
assert first_word('3M Privacy Filter') == '3M'

df['brand'] = df['Title'].apply(extract_brand, known=known).str.title()
# Product lines and spelling variants of one maker -> the maker. Separate brands
# that shoppers see on their own (Alienware, Beats, Xbox, Ring, Blink, eero,
# Soundcore, Tapo, HyperX, ...) stay separate, per the Amazon-brand rule.
SUB_BRANDS = {'Wd_Black': 'Western Digital', 'Wd': 'Western Digital', 'Hewlett-Packard': 'Hp',
              'Rog': 'Asus', 'Tuf': 'Asus', 'Legion': 'Lenovo', 'Predator': 'Acer',
              'Joy-Con': 'Nintendo', 'Oculus': 'Meta'}
df['brand'] = df['brand'].replace(SUB_BRANDS)
print((df.drop_duplicates('Title')['brand'] == 'Unbranded').sum(), 'products Unbranded')
df['brand'].value_counts().head(20)

40 products Unbranded


brand
Asurion      3067
Hp           1975
Sony          969
Dell          927
Lenovo        883
Samsung       869
Canon         792
Duracell      698
Ugreen        694
Apple         645
Energizer     599
Avery         586
Epson         577
Fujifilm      565
Kodak         558
Asus          555
Pyle          548
Belkin        525
Nikon         473
Logitech      461
Name: count, dtype: int64

## Drop Asurion protection plans

(Also catches one non-Asurion bundle plan, `Complete Protect`, same reasoning.)

Every `Protection Plan` row is an Asurion extended-warranty add-on (e.g.
`"ASURION 2 Year Laptop Protection Plan ($2000 - $2999.99)"`) -- 3,067 listing
rows / 665 distinct titles, 7.5% of the distinct-product catalog. Dropped
entirely because they carry no usable signal for any question this dataset is
being used for, and actively distort the ones they touch:

- **No price, no sales, never available:** 0% in stock, 0 rows with
  `bought_in_last_month`, only 5/665 titles have any price at all. There is no
  active offer to model.
- **Never Amazon's Choice, never sponsored:** 665 guaranteed-negative titles
  would drag down the badge base rate and add pure-noise negatives to any model
  of `is_amazons_choice`.
- **Not a consumer product choice:** a warranty tier is bought as an add-on to
  another product, not chosen on its own merits -- it doesn't belong in
  brand/price/rating comparisons, and it made Asurion the single "largest
  brand" in the catalog, inflating the brand-size tail.
- **Ratings/reviews don't look like they belong to the plan:** spot-checking a
  few titles against live Amazon listings didn't match (e.g. 1 star / no
  reviews live vs ~4 stars here) -- whatever these numbers describe, it isn't
  reliably the plan itself.

The `Protection Plan` category pattern is kept in `CATEGORY_PATTERNS` so these
rows are identified here (and not misfiled into another category first).

In [14]:
plans = df['category'] == 'Protection Plan'
assert not (df.loc[~plans, 'brand'] == 'Asurion').any()
print('non-Asurion plans:', df.loc[plans & (df['brand'] != 'Asurion'), 'Title'].unique())
print(f"dropping {plans.sum()} protection-plan rows ({df.loc[plans, 'Title'].nunique()} titles): "
      f"in_stock={df.loc[plans, 'in_stock'].mean():.0%}, "
      f"has bought_in_last_month={df.loc[plans, 'bought_in_last_month'].notna().sum()}, "
      f"titles with any price={df.loc[plans & df['current_price'].notna(), 'Title'].nunique()}, "
      f"amazons_choice={df.loc[plans, 'is_amazons_choice'].sum()}")
df = df[~plans].reset_index(drop=True)

non-Asurion plans: <ArrowStringArray>
['Complete Protect: One plan covers all eligible past and future purchases on Amazon']
Length: 1, dtype: str
dropping 3068 protection-plan rows (666 titles): in_stock=0%, has bought_in_last_month=0, titles with any price=6, amazons_choice=0


## Drop non-electronics

Products matched by `NON_ELECTRONICS` (see the category section): bags,
toys and puzzles, personal care, car care, pet, kitchen, furniture, hand
tools, apparel. ~115 distinct products, ~1.4% of the catalog.

**Plus plain stationery (user's call, 2026-09-25):** binders, file folders,
pens, pencils, markers, sticky notes, staplers, whiteboards, envelopes, index
cards, dividers (58 products: 55 by the rule, 3 whose reference
label said Office, caught by `to_current`). Office & School Supplies had
become half stationery, e.g. a $2.49 binder in a category of $40 calculators
and label makers. Kept: the devices, their supplies (label tapes, laminating
pouches, binding combs) and printable paper/cards/labels (printing supplies,
like photo paper in Printers & Ink). The dropped list is saved to
`data/dropped_stationery.csv`.

**Why this is a scope decision, not selection bias:** the rule is decided
from the product type in the title alone, *before* looking at any outcome
(badge, sales, rating). A drop biases results when the criterion depends on
the outcome (e.g. dropping low sellers); this one defines the population
instead -- every downstream claim is about *consumer electronics and
accessories in this scrape*, which is what the dataset is overwhelmingly
about anyway.

**Checks below:**
1. **Does the rule match a hand review?** Before writing the rule, the
   non-electronics were identified by reading titles directly
   (`data/non_electronics_handmarked.csv`, 114 products). The rule should
   recover nearly all of them, and anything it flags beyond that list should
   be defensible.
2. **Do the dropped products differ on the outcomes?** Compared at the
   distinct-product level. Expected: same Amazon's Choice and sponsorship
   rates (so the badge model is unaffected), but cheaper and higher-volume
   (commodity goods) -- which is the reason to exclude them from electronics
   price tiers and the sales model rather than a reason to keep them.

**Still to do: sensitivity check** -- refit the final model with these
products included and confirm conclusions don't change (tracked in
`HANDOFF.md`).

In [15]:
non_elec = df['category'] == 'Non-Electronics'
stationery = non_elec & df['Title'].map(is_stationery)
(df.loc[stationery].drop_duplicates('Title')[['Title', 'brand', 'current_price']]
   .sort_values('Title').to_csv('../data/dropped_stationery.csv', index=False))
print(f"plain stationery (user's call): {df.loc[stationery, 'Title'].nunique()} products -> data/dropped_stationery.csv")

# 1. rule vs. hand-marked list
hand = set(pd.read_csv('../data/non_electronics_handmarked.csv')['Title'])
flagged = set(df.loc[non_elec, 'Title'])
print(f'hand-marked: {len(hand)} | rule flags: {len(flagged)} | '
      f'hand-marked caught: {len(hand & flagged)} ({len(hand & flagged) / len(hand):.0%})')
print('\nhand-marked, not flagged (and where they landed instead):')
missed = df[df['Title'].isin(hand - flagged)].drop_duplicates('Title')
print(missed[['Title', 'category']].to_string(index=False, max_colwidth=70))
print('\nflagged, not hand-marked:')
print('\n'.join(sorted(t[:80] for t in flagged - hand)))

plain stationery (user's call): 55 products -> data/dropped_stationery.csv
hand-marked: 114 | rule flags: 171 | hand-marked caught: 110 (96%)

hand-marked, not flagged (and where they landed instead):
                                                                 Title     category
Burt's Bees Back To School Dorm Essentials, Cucumber & Mint Face Wi...        Other
Kipling Women's Seoul Small Tablet Backpack, Durable, Padded Should... Bags & Cases
                              JansSport Pack BIG STUDENT GRAPHITE GREY        Other
                               Sprayground PACMAN GHOST GANG SHARMOUTH        Other

flagged, not hand-marked:
Adidas Unisex-Adult Interval Reversible Wristband – Athletic Sweatband for Men a
Amazon Basics 10 Security-Tinted Self-Seal Business Letter Envelopes, Peel & Sea
Amazon Basics Aluminum Storage Clipboard, Two-Tier, Standard Clip, 14" x 9", Sil
Amazon Basics Blank Index Cards, White, 3" x 5", 1000 Count (10 Pack of 100)
Amazon Basics Classic Lined Note

In [16]:
# 2. do they differ on the outcomes? (one row per distinct product; badge = seen on any scrape)
per_product = (
    df.sort_values('collected_at')
      .groupby('Title')
      .agg(non_electronics=('category', lambda s: s.iloc[-1] == 'Non-Electronics'),
           amazons_choice=('is_amazons_choice', 'any'), sponsored=('is_sponsored', 'any'),
           final_price=('final_price', 'last'), number_of_reviews=('number_of_reviews', 'last'),
           rating=('rating', 'last'), bought_in_last_month=('bought_in_last_month', 'last'))
)
g = per_product.groupby('non_electronics')
pd.DataFrame({
    'products': g.size(),
    "Amazon's Choice rate": g['amazons_choice'].mean().round(3),
    "Amazon's Choice count": g['amazons_choice'].sum(),
    'sponsored rate': g['sponsored'].mean().round(3),
    'median price': g['final_price'].median(),
    'median reviews': g['number_of_reviews'].median(),
    'mean rating': g['rating'].mean().round(2),
    'median bought/month': g['bought_in_last_month'].median(),
}).rename(index={False: 'electronics (kept)', True: 'non-electronics (dropped)'}).T

non_electronics,electronics (kept),non-electronics (dropped)
products,7971.000,171.000
Amazon's Choice rate,0.035,0.047
Amazon's Choice count,279.000,8.000
sponsored rate,0.044,0.029
median price,76.965,18.100
median reviews,902.000,6740.000
mean rating,4.450,4.600
median bought/month,400.000,1000.000


In [17]:
print(f"dropping {non_elec.sum()} listing rows ({df.loc[non_elec, 'Title'].nunique()} distinct products)")
df = df[~non_elec].reset_index(drop=True)
df['category'].value_counts()

dropping 821 listing rows (171 distinct products)


category
Audio                              6117
Computers & Laptops                5576
Cameras & Photography              4335
Printers & Ink                     2962
Cables & Charging                  1956
Batteries                          1727
Phone & Tablet Accessories         1653
Monitors & Displays                1604
Networking                         1461
Other                              1356
Smart Home & Wearables             1349
Keyboards & Mice                   1220
Storage & Memory Cards             1163
Gaming                             1083
Office & School Supplies            812
Optics                              688
Car Electronics                     679
Phones & Tablets                    479
TV & Home Entertainment             442
Power Strips & Surge Protectors     428
Marine & GPS                        402
Bags & Cases                        332
Name: count, dtype: int64

## Reference labels (hand and LLM)

Better labels than the keyword rules exist for ~1,000 products. They're
used twice: to **train** the category model below, and to **replace** its
output where they exist. Sources, highest priority first:

1. **User** -- hand labels: 250 random products
   (`labeling/random_250_labels.csv`) and 10 hard cases
   (`labeling/sonnet_unsure_47_labels.csv`). A few obvious mis-keys in the
   250 were fixed and noted in the file (raw export kept as
   `random_250_labels_raw.csv`).
2. **Gold** -- the 160 hand-checked products (`data/category_gold_sample.csv`).
3. **Sonnet (confident)**, then 4. **Sonnet (unsure)** -- Claude Sonnet
   agents labelled 604 products the pipeline was most likely to get wrong
   (then-embedding overrides, `Other`, within-category price outliers), blind
   to the pipeline label and following `labeling/guidelines.md`. Checked
   against the gold 160: 96% agreement (98% when confident).

Gold and Sonnet labels were made before Revision 4, so the Revision 4 rules
are re-applied to them (`to_current`: a device labelled "Phone & Tablet
Accessories" becomes "Phones & Tablets", etc.).


In [18]:
def to_current(title, label):
    """Re-apply the Revision 4 rules to a label made before them."""
    t = title.lower()
    if label == 'Phone & Tablet Accessories' and re.search(PHONES, t):
        return 'Phones & Tablets'
    if label in ('Phone & Tablet Accessories', 'Monitors & Displays') and re.search(WATCHES, t):
        return 'Smart Home & Wearables'
    if label == 'Computers & Laptops' and categorize(title) == 'Storage & Memory Cards':
        return 'Storage & Memory Cards'
    if label in CAR_FROM and CAR.search(title):
        return 'Car Electronics'
    if label in ('Cables & Charging', 'Other') and re.search(VIDEO, t):  # not Computers: laptops list a webcam
        return 'Cameras & Photography'
    if label == 'Audio' and re.search(VIDEO_SYSTEM, t):
        return 'Cameras & Photography'
    if label != 'Power Strips & Surge Protectors' and categorize(title) == 'Power Strips & Surge Protectors' and re.search(UPS, t):
        return 'Power Strips & Surge Protectors'  # UPS units (user's call, 2026-09-25)
    if label == 'Office & School Supplies' and is_stationery(title):
        return 'Non-Electronics'  # plain stationery (user's call, 2026-09-25)
    return label

user = pd.concat([pd.read_csv('../labeling/random_250_labels.csv'),
                  pd.read_csv('../labeling/sonnet_unsure_47_labels.csv')])
gold = pd.read_csv('../data/category_gold_sample.csv')
sonnet = pd.read_csv('../labeling/sonnet_labels.csv')
r250 = pd.read_csv('../labeling/random_250_labels.csv')
refs = pd.concat([
    user.assign(source='user', label=user['true_category'],
                eval_set=np.where(user['Title'].isin(r250['Title']) & (user['verdict'] != 'unsure'), 'random_250', '')),
    gold.assign(source='gold', label=gold['true_category'], eval_set='gold_' + gold['sample'].astype(str)),
    sonnet.assign(source=np.where(sonnet['confident'], 'sonnet_confident', 'sonnet_unsure'), label=sonnet['sonnet'], eval_set=''),
])[['Title', 'source', 'eval_set', 'label']]
refs['label_now'] = [to_current(t, l) for t, l in zip(refs['Title'], refs['label'])]
refs.to_csv('../data/category_reference_labels.csv', index=False)  # read by category_check.ipynb
# user labels already follow the current rules; show any the re-application would change
print(refs.query("source == 'user' and label != label_now")[['Title', 'label', 'label_now']].to_string())
assert set(refs['label_now']) <= set(df['category']) | {'Non-Electronics'}

priority = ['user', 'gold', 'sonnet_confident', 'sonnet_unsure']
ref = (refs.assign(p=refs['source'].map(priority.index)).sort_values('p')
           .drop_duplicates('Title').set_index('Title'))


                                                                                                                                                                                                     Title                       label                        label_now
40                         AT&T BL102-2 DECT 6.0 2-Handset Cordless Phone for Home with Answering Machine, Call Blocking, Caller ID Announcer, Audio Assist, Intercom, and Unsurpassed Range, Silver/Black  Phone & Tablet Accessories                 Phones & Tablets
66                                                                                    WD_BLACK 8TB SN850X NVMe Internal Gaming SSD Solid State Drive - Gen4 PCIe, M.2 2280, Up to 7,200 MB/s - WDS800T2X0E         Computers & Laptops           Storage & Memory Cards
86                                                            Seagate Storage Expansion Card 2TB Solid State Drive - NVMe SSD for Xbox Series X|S, Quick Resume, Plug & Play, Licensed (STJR2000400) Black      

## Category model: title embedding + keyword rule -> logistic regression

The keyword rules are good but systematically wrong in places (renewed
iPhones as accessories, before Revision 4). One logistic regression makes
the final automatic call (`category_auto`):

- **Inputs:** the sentence embedding of the title (the part before `for` /
  `compatible with` / first comma or bracket -- compatibility text is what
  misleads; `all-MiniLM-L6-v2`, small, local, deterministic) **plus the
  keyword rule's label** as a one-hot feature.
- **Targets:** the *corrected* labels -- the reference label where one
  exists, otherwise the keyword label. So the model learns what categories
  look like from ~8,000 mostly-right labels, and from the ~1,000 reference
  labels it learns *when the rules are wrong* (e.g. "iPhone ... (Renewed)"
  with rule label "accessories" -> actually a phone).
- **Out-of-fold** (5-fold): each product's prediction comes from a model that
  never saw that product's own label, so the reference labels don't leak
  into the accuracy check below, and a wrong keyword label can't just be
  memorised.

This replaces an earlier two-opinion design (logistic regression trained on
the raw keyword labels + zero-shot similarity to category descriptions,
overriding only where both agreed). Head-to-head on held-out labels (models
trained without the test products; user's suggestion):

| setup | random 250 | gold sample 2 |
|---|---|---|
| keyword rules only | 94.4% | 91.2% |
| old: rules + "both agree" override | 94.8% | 91.2% |
| LR on embeddings, corrected labels | 93.2% | 95.0% |
| LR on the ~770 reference labels only | 76.8% | 85.0% |
| **LR on embeddings + rule label, corrected labels (used)** | **95.2%** | **92.5%** |

The gain is ~1 point -- within noise at 330 test products -- so the case for
it is that it's simpler (one model, no hand-written category descriptions,
no agreement rule) and learns from our best labels, not that it's clearly
more accurate. The reference labels alone are too few (22 categories, ~770
mostly-hard examples) to replace the rules. Settings (`C=10`, rule feature
x3) were not tuned on the test sets.

The keyword label is kept as `category_regex`, so every change is traceable.


In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

# category is a function of Title, so work on distinct titles
titles = df.drop_duplicates('Title').set_index('Title')['category']
head = titles.index.str.split(r'\b(?:for|compatible with|works with)\b|[,|(\[]', n=1, regex=True).str[0].str.strip()
head = pd.Series(head, index=titles.index).where(lambda s: s.str.len() >= 15, titles.index.to_series())

model = SentenceTransformer('all-MiniLM-L6-v2')
emb = model.encode(head.tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
cats = np.array(sorted(titles.unique()))
rule = (titles.to_numpy(dtype=object)[:, None] == cats[None, :]).astype(float)
X = np.hstack([emb, 3 * rule])  # x3: rough scale match to the unit-length embedding, untuned

y = titles.index.to_series().map(ref['label_now']).fillna(titles).to_numpy(dtype=object)  # corrected labels
trainable = y != 'Non-Electronics'  # too few to learn; those products get dropped below anyway
pred = np.empty(len(titles), dtype=object)
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    tr = tr[trainable[tr]]
    pred[te] = LogisticRegression(C=10, max_iter=3000).fit(X[tr], y[tr]).predict(X[te])

changed = pred != titles.to_numpy(dtype=object)
print(f'model changed the keyword label for {changed.sum()} of {len(titles)} products ({changed.mean():.1%})')
df['category_regex'] = df['category']
df['category_auto'] = df['Title'].map(pd.Series(pred, index=titles.index))
pd.crosstab(titles[changed], pred[changed])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model changed the keyword label for 284 of 7971 products (3.6%)


col_0,Audio,Bags & Cases,Cables & Charging,Cameras & Photography,Car Electronics,Computers & Laptops,Keyboards & Mice,Marine & GPS,Networking,Office & School Supplies,Optics,Phone & Tablet Accessories,Power Strips & Surge Protectors,Printers & Ink,Smart Home & Wearables,Storage & Memory Cards,TV & Home Entertainment
category,,,,,,,,,,,,,,,,,
Audio,0,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,4
Batteries,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Cables & Charging,5,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
Cameras & Photography,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0
Computers & Laptops,23,0,13,3,0,0,3,0,3,0,2,1,0,0,0,1,4
Gaming,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4,0
Marine & GPS,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0
Monitors & Displays,12,0,6,0,5,0,0,2,0,0,0,3,0,0,3,0,0
Networking,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,4


## Label corrections

Where a reference label exists it replaces the model's output. Products
relabelled `Non-Electronics` are dropped, same as the rule-based drop.

Columns: `category_regex` (keyword rules), `category_auto` (the model -- the
fully automatic label; measure accuracy on this), `category` (final),
`category_source` (which reference set `category`; `pipeline` = none).


In [20]:
df['category'] = df['Title'].map(ref['label_now']).fillna(df['category_auto'])
df['category_source'] = df['Title'].map(ref['source']).fillna('pipeline')
per_title = df.drop_duplicates('Title')
changed = per_title['category'] != per_title['category_auto']
print(f"{changed.sum()} of {len(per_title)} distinct products relabelled, by source:")
print(per_title[changed]['category_source'].value_counts().to_string())


256 of 7971 distinct products relabelled, by source:
category_source
sonnet_confident    190
sonnet_unsure        39
user                 18
gold                  9


In [21]:
# Accuracy of the automatic labels against the reference labels (distinct
# products; "unsure" user verdicts excluded). Caveat: the Revision 4 rules
# were written after seeing the random 250's errors, so its score is now
# optimistic; before Revision 4 it scored 87.6%, the independent number.
# Gold sample 2 wasn't used for any revision.
evals = {name: refs[refs['eval_set'] == key] for name, key in
         [('gold sample 1 (tuned on)', 'gold_1'), ('gold sample 2 (held out)', 'gold_2'), ('random 250 (user)', 'random_250')]}
auto = per_title.set_index('Title')[['category_regex', 'category_auto']]
rows = []
for name, e in evals.items():
    truth = e.set_index('Title')['label_now']
    a = auto.reindex(truth.index)
    rows.append({'eval set': name, 'n': len(truth),
                 'keyword rules': (a['category_regex'] == truth).mean(),
                 'model (category_auto)': (a['category_auto'] == truth).mean()})
pd.DataFrame(rows).set_index('eval set').round(3)


,n,keyword rules,model (category_auto)
eval set,,,
gold sample 1 (tuned on),80,0.938,0.938
gold sample 2 (held out),80,0.925,0.938
random 250 (user),250,0.928,0.936


In [22]:
relabelled_out = df['category'] == 'Non-Electronics'
# stationery that reached Office via a reference label rather than the rules: add to the log
late = df.loc[relabelled_out & df['Title'].map(is_stationery)].drop_duplicates('Title')[['Title', 'brand', 'current_price']]
log = pd.concat([pd.read_csv('../data/dropped_stationery.csv'), late]).drop_duplicates('Title').sort_values('Title')
log.to_csv('../data/dropped_stationery.csv', index=False)
print(f'stationery log: {len(log)} products ({len(late)} caught at the reference-label step)')
print(f"dropping {relabelled_out.sum()} rows ({df.loc[relabelled_out, 'Title'].nunique()} products) relabelled Non-Electronics")
df = df[~relabelled_out].reset_index(drop=True)
df.drop_duplicates('Title')['category'].value_counts()


stationery log: 58 products (3 caught at the reference-label step)
dropping 623 rows (25 products) relabelled Non-Electronics


category
Audio                              1205
Computers & Laptops                1152
Printers & Ink                      910
Cameras & Photography               727
Phone & Tablet Accessories          417
Cables & Charging                   378
Monitors & Displays                 332
Storage & Memory Cards              331
Batteries                           321
Networking                          315
Keyboards & Mice                    299
Office & School Supplies            266
Smart Home & Wearables              227
TV & Home Entertainment             205
Phones & Tablets                    186
Gaming                              155
Car Electronics                     146
Power Strips & Surge Protectors     139
Bags & Cases                        122
Marine & GPS                         52
Optics                               48
Other                                13
Name: count, dtype: int64

In [23]:
df = df.drop(columns=['image_url'])
df['collected_at'] = pd.to_datetime(df['collected_at'], format='%d-%m-%Y %H:%M')
df['collected_at'].describe()


count                         37201
mean     2025-08-26 11:53:16.073761
min             2025-08-21 11:14:00
25%             2025-08-24 21:57:00
50%             2025-08-25 11:25:00
75%             2025-08-29 11:22:00
max             2025-08-30 19:56:00
Name: collected_at, dtype: object

## Final check + save

## Sanity checks (no changes, just verification)

**Numeric extremes:** checked the top/bottom of `current_price` and top of
`number_of_reviews` against their `Title` to rule out parsing errors. All
plausible -- top prices are real high-end laptops/cameras (ASUS ROG, Canon
EOS R5, Sony Alpha), bottom prices are real cheap commodity items (button
batteries, binder clips), top review counts are genuinely mass-market
Amazon Basics/SanDisk products. No fixes needed.

**Missing rating/reviews (1,024 rows, 2.5%):** same "structural, not random"
pattern as `current_price`. These rows are far more likely to also be
missing `bought_in_last_month` (92.6% vs 23.5% for rated rows) and are more
often sponsored (33% vs 15%) -- consistent with brand-new/rarely-purchased
listings that haven't accumulated reviews yet, not a scraping glitch. Left
as NaN for the same reason as `current_price`: no reliable value to impute.

In [24]:
print(df.nlargest(5, 'current_price')[['Title', 'current_price']].to_string())
print()
print(df.nsmallest(5, 'current_price')[['Title', 'current_price']].to_string())
print()
print(df.nlargest(5, 'number_of_reviews')[['Title', 'number_of_reviews']].to_string())
print()

missing = df['rating'].isna()
print('rating missing:', missing.sum(), '/', len(df))
print('bought_in_last_month NaN rate -- missing-rating:', df.loc[missing, 'bought_in_last_month'].isna().mean(),
      ' has-rating:', df.loc[~missing, 'bought_in_last_month'].isna().mean())
print('is_sponsored rate -- missing-rating:', df.loc[missing, 'is_sponsored'].mean(),
      ' has-rating:', df.loc[~missing, 'is_sponsored'].mean())

                                                                                                                                                                                                          Title  current_price
8677                                                                                                                                  Logitech Rally bar and tap IP Bundle, 1920 Pixels x 1080 Pixels, Graphite        4699.00
5849   ASUS ROG Strix SCAR 18 (2025) Gaming Laptop, 18” ROG Nebula HDR 16:10 2.5K 240Hz/3ms, NVIDIA GeForce RTX 5090, Intel Core Ultra 9 275HX, 32GB DDR5, 2TB PCIe Gen 4 SSD, Wi-Fi 7, Win 11 Pro, G835LX-XS97        4399.00
9321                                   Canon EOS R3 Mirrorless Camera (Body Only), Full-Frame Camera, 6K 60p RAW, 4K 120p Video, 24.1 Megapixel CMOS Sensor, DIGIC X Image Processor, for Sports, Action, Black        4399.00
5091                                                                                                        

In [25]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df.to_csv('../data/amazon_sales_data_cleaned.csv', index=False)


(37201, 27)
Title                                  str
bought_in_last_month               float64
is_best_seller                        bool
is_sponsored                          bool
delivery_details                       str
product_url                            str
collected_at                datetime64[us]
rating                             float64
number_of_reviews                  float64
current_price                      float64
listed_price                       float64
has_coupon                            bool
coupon_pct                         float64
coupon_amount                      float64
final_price                        float64
in_stock                              bool
is_amazons_choice                     bool
deal_pct                           float64
has_deal_badge                        bool
is_small_business                     bool
has_sustainability_badge              bool
bought_upper                       float64
category                               str

## Distinct-product view for catalog-level analysis

`amazon_sales_data_cleaned.csv` is a table of **listing-observations**, not
distinct products: 365 titles (0.9% of unique titles) account for 33,248 of
42,675 raw rows (79.7%). E.g. one Duracell 9V battery listing was scraped
~744 times, roughly every minute across all 6 collection days, with `rating`
and `number_of_reviews` frozen identically every time -- almost certainly a
recurring sponsored/carousel ad slot being logged as if it were hundreds of
independent listings, not organic catalog diversity.

This means **any row-level aggregate on the main cleaned file** (category
counts, brand rankings, average price/rating by category) is dominated by
which items happened to occupy an ad slot most often during the scrape, not
by the actual composition of the catalog. That's a legitimate thing to
study on its own (e.g. "what does Amazon keep re-showing"), but it is very
easy to mistake for catalog composition if you don't collapse it first.

`product_url` can't be used to de-duplicate: 5,492 rows (13%) are one-time
`/sspa/click?...` ad-redirect tokens that are unique per impression by
construction, never repeating even for the same physical product scraped
twice. `Title` is the best available (imperfect) product-identity proxy, so
`amazon_products_deduped.csv` below collapses to one row per `Title`, keeping
the most recent scrape. Use this file for anything about catalog composition
or "typical" products; use the main listing-level file for anything about
scrape/impression frequency itself.

**`product_family`.** Many distinct titles are variants of one product: HP
sells each toner colour and yield as its own listing, Apple Watch sizes and
colours are separate, some products are re-listed with a slightly different
title. They're kept as separate products -- each is its own listing, with
its own price and sales, and Amazon's Choice usually goes to *one* variant
(in families that have a badge, only ~28% of members carry it) -- but
they're not independent: sales correlate strongly within a family (ICC of
log sales ~0.6, of log price ~0.9). `product_family` groups them (same
brand, title-embedding cosine >= 0.95), so models can add a family level
under brand instead of counting HP's 59 toner listings as 59 independent
pieces of evidence about HP.


## Badge rate (`badge_scrapes`, `badge_hits`, `badge_rate`, `badge_rate_pooled`)

`is_amazons_choice` on the deduped file is just whichever scrape came last,
and a badge flickers minute to minute (see the daily section below), so the
product-level badge measure is the **rate**: badged scrapes / scrapes.
Unlike "ever badged", it isn't inflated by scrape count -- a product scraped
200 times has more chances to be caught badged, but not a higher share.

Most products were scraped once, so their raw `badge_rate` is 0 or 1.
`badge_rate_pooled` shrinks each rate toward the typical rate in proportion to
how little data it has (Beta-Binomial fitted across all products): one badged
scrape out of one reads ~11%, while 35 of 269 (Energizer MAX AA) barely moves
from 13%. Use the pooled rate for ranking/plots; the Bayesian model works
from `badge_hits` / `badge_scrapes` directly.

**Why some rates are so low -- two explanations we can't separate:**
1. *Amazon*: the badge is assigned per search query and re-assigned through
   the day, so a product only wears it in some contexts.
2. *The scraper*: the badge may simply not be captured on some scrapes (page
   layout, caching, a render race). Evidence this is at least part of it:
   after a badged scrape, the next scrape of the same product ~1 minute
   later is unbadged 92% of the time with stock, price and sponsorship
   unchanged; and within a product the badge shows on ~33% of sponsored
   scrapes vs ~9% of organic ones, which smells like the ad card and the
   organic card rendering it differently.

So a low rate means "rarely *seen* with the badge", not necessarily "rarely
badged". Treat it as a relative measure between products, not an absolute
share of time badged.


In [26]:
products = (
    df.sort_values('collected_at')
      .drop_duplicates(subset='Title', keep='last')
      .reset_index(drop=True)
)
print(f'{len(df)} listing-observations -> {len(products)} distinct products (by Title)')

# product_family: near-identical listings of the same brand (colour / capacity /
# pack-size variants, "Renewed" vs "Renewed Premium", re-listed titles).
# Same brand + title-embedding cosine >= 0.95, joined transitively.
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
E = model.encode(products['Title'].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
b = products['brand'].to_numpy(dtype=object)
_, products['product_family'] = connected_components(csr_matrix((E @ E.T >= 0.95) & (b[:, None] == b[None, :])), directed=False)
fam_size = products['product_family'].map(products['product_family'].value_counts())
print(f"{(fam_size > 1).sum()} products in {products.loc[fam_size > 1, 'product_family'].nunique()} multi-listing families; largest {fam_size.max()}")
# Badge rate: share of a product's scrapes that showed Amazon's Choice (see markdown).
from scipy import stats, optimize
hits = df.groupby('Title')['is_amazons_choice'].agg(badge_scrapes='size', badge_hits='sum')
products = products.join(hits, on='Title')
products['badge_rate'] = products['badge_hits'] / products['badge_scrapes']
# Pooled: Beta-Binomial fitted across products (empirical Bayes), so a single
# scrape is pulled toward the typical rate instead of reading 0% or 100%.
# ponytail: one global prior; the PyMC model adds brand/category levels.
k, n = products['badge_hits'], products['badge_scrapes']
a, b_ = np.exp(optimize.minimize(lambda p: -stats.betabinom.logpmf(k, n, *np.exp(p)).sum(), [0, 3]).x)
products['badge_rate_pooled'] = (k + a) / (n + a + b_)
print(f"per-scrape badge rate {k.sum() / n.sum():.2%}; Beta prior a={a:.2f}, b={b_:.1f} (mean {a / (a + b_):.2%})")
print(f"ever badged: {(k > 0).sum()} products; of {(n >= 20).sum()} with 20+ scrapes, {((n >= 20) & (k == 0)).sum()} never showed it")
assert products['badge_rate_pooled'].between(0, 1).all() and (products['badge_hits'] <= products['badge_scrapes']).all()
products.to_csv('../data/amazon_products_deduped.csv', index=False)
products['category'].value_counts()


37201 listing-observations -> 7946 distinct products (by Title)


1606 products in 594 multi-listing families; largest 19
per-scrape badge rate 1.87%; Beta prior a=0.20, b=9.7 (mean 2.01%)
ever badged: 276 products; of 292 with 20+ scrapes, 194 never showed it


category
Audio                              1205
Computers & Laptops                1152
Printers & Ink                      910
Cameras & Photography               727
Phone & Tablet Accessories          417
Cables & Charging                   378
Monitors & Displays                 332
Storage & Memory Cards              331
Batteries                           321
Networking                          315
Keyboards & Mice                    299
Office & School Supplies            266
Smart Home & Wearables              227
TV & Home Entertainment             205
Phones & Tablets                    186
Gaming                              155
Car Electronics                     146
Power Strips & Surge Protectors     139
Bags & Cases                        122
Marine & GPS                         52
Optics                               48
Other                                13
Name: count, dtype: int64

## (Title, day) daily aggregation -- for Amazon's Choice modeling

`is_amazons_choice` is not stable across repeated scrapes of the same title:
154/1,074 multi-scrape titles (14%) show it flip between `True`/`False`.
Checked *when* it flips before trusting this as signal: **145 of those 154
(94%) flip within the same day**, often minutes apart, with `in_stock`,
`current_price`, and `rating` all identical between the flipping rows (e.g.
one title goes `False` -> `True` in 6 minutes with every other field
unchanged). Amazon does not award/revoke the badge in 6 minutes with nothing
else about the listing changing -- this is intermittent rendering/capture
noise (caching, personalization, or a scrape race condition), not a real
event. Only 9/154 (6%) flip exclusively across different days, which is far
more consistent with a genuine change.

Neither the raw listing-level file (treats capture noise as real signal) nor
`amazon_products_deduped.csv` (arbitrarily picks one snapshot and could pick
either side of a same-day flicker) is the right unit for modeling this
target. Instead, aggregate to one row per (`Title`, day): `is_amazons_choice`
is set to `True` if the badge was observed **at least once** that day, which
denoises the within-day capture noise (a missed render one minute doesn't
erase a real same-day badge) while preserving genuine day-to-day changes,
since those happen at the day boundary this aggregation respects. `n_scrapes`
and `n_choice_observed` are kept per row so this decision is inspectable
and reversible (e.g. for a Beta-Binomial treatment of same-day repeat
observations later) rather than silently collapsed away.

In [27]:
daily = (
    df.assign(collected_date=df['collected_at'].dt.date)
      .groupby(['Title', 'collected_date'], as_index=False)
      .agg(
          category=('category', 'first'),
          brand=('brand', 'first'),
          n_scrapes=('is_amazons_choice', 'size'),
          n_choice_observed=('is_amazons_choice', 'sum'),
          is_amazons_choice=('is_amazons_choice', 'any'),
          in_stock=('in_stock', 'any'),
          is_sponsored=('is_sponsored', 'any'),
          has_coupon=('has_coupon', 'any'),
          current_price=('current_price', 'median'),
          listed_price=('listed_price', 'median'),
          final_price=('final_price', 'median'),
          rating=('rating', 'median'),
          number_of_reviews=('number_of_reviews', 'median'),
          bought_in_last_month=('bought_in_last_month', 'median'),
      )
)

print(f'{len(df)} listing-observations -> {len(daily)} (Title, day) rows')
print('is_amazons_choice rate:', daily['is_amazons_choice'].mean(), f"({daily['is_amazons_choice'].sum()} rows)")
print('rows with >1 scrape that day:', (daily['n_scrapes'] > 1).sum())
daily.to_csv('../data/amazon_choice_daily.csv', index=False)
daily.head()


37201 listing-observations -> 8351 (Title, day) rows
is_amazons_choice rate: 0.03556460304155191 (297 rows)
rows with >1 scrape that day: 1011


,Title,collected_date,category,brand,n_scrapes,n_choice_observed,is_amazons_choice,in_stock,is_sponsored,has_coupon,current_price,listed_price,final_price,rating,number_of_reviews,bought_in_last_month
0,"(2) Kicker 40PS692 6x9"" 180w Polaris/ATV/UTV/R...",2025-08-24,Audio,Kicker,2,0,False,True,False,False,169.96,169.96,169.96,4.2,68.0,100.0
1,10 Duracell 357 303 A76 PX76 SR44W/SW LR44 AG1...,2025-08-24,Batteries,Duracell,1,0,False,True,False,False,19.17,19.17,19.17,4.7,1088.0,500.0
2,10-18mm F2.8 DC DN (for E Mount),2025-08-24,Cameras & Photography,Unbranded,1,0,False,True,False,False,693.41,729.00,693.41,4.7,141.0,50.0
3,100 Count Energizer Industrial EN92 Alkaline A...,2025-08-21,Batteries,Energizer,1,0,False,True,False,False,41.00,48.99,41.00,4.6,1855.0,300.0
4,100 count Energizer Industrial AA EN91 Alkalin...,2025-08-21,Batteries,Energizer,1,0,False,True,False,False,47.99,55.99,47.99,4.7,8073.0,500.0
